# The Anatomy of AI · Lecture 9 · An Atlas of Everything It Sees — the notebook

Lecture 8 drew one neuron at a time. Here, as in the lecture, you label every neuron by machine, take one photo apart
into a few building blocks, lay a layer's activations out as one map, and use that map to find a shortcut: a baseball
that pushes a whale toward "shark". Step by step:

1. **Network Dissection:** score every neuron of four layers against 2,000 photos with every pixel labelled;
2. **Building blocks:** factorise one photo's layer into six recurring mixes, draw them, and measure who made the answer;
3. **an Activation Atlas,** smaller than the lecture's so it fits on free Colab;
4. **a whale-or-shark class atlas,** also smaller, and where baseballs land on it;
5. **the patch attack,** at the lecture's full scale: a baseball pasted on 1,110 grey whales.

Each section names the lecture plate it follows. Wherever the lecture gives a number, a `check(...)` line compares
yours with it and prints ✓; if a check fails, the notebook stops. The two atlases here use fewer activations than the
lecture's (it laid out 100,000 and 20,000); their checks compare what does not depend on size (what the map is made of,
and the lecture's claims about it) and say so.

**Runs on:** free Colab with a T4 GPU (Runtime → Change runtime type → T4), in about fifteen minutes. It downloads the
model (27 MB), the lecture's copy of the ImageNet test photos (two files of about 350 MB), ADE20K (970 MB, of which it
reads the 2,000 test photos) and two of the forty ImageNet training files (the grey whales and the great white
sharks, about 450 MB each).

## 0. Setup
Installs (pinned), the plot style, and `check()`.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3", "scikit-learn==1.5.2", "umap-learn==0.5.7", "pynndescent==0.5.13", "numba==0.61.0", "llvmlite==0.44.0")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", "a GPU" if DEV == "cuda" else "the CPU")
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

In [ ]:
LECTURE = json.loads(r'''{"channels_total": 5804, "dissect": {"mixed3b": {"channels": 480, "detectors": 29, "concepts": 13, "top": [[193, "grass", 0.108], [175, "skyscraper", 0.1], [434, "skyscraper", 0.1]], "label_55": null, "iou_55": null, "counts_top": ["skyscraper", 7]}, "mixed4c": {"channels": 512, "detectors": 182, "concepts": 33, "top": [[447, "car", 0.261], [143, "car", 0.198], [274, "car", 0.155]], "label_55": null, "iou_55": null, "counts_top": ["person", 49]}, "mixed4e": {"channels": 832, "detectors": 322, "concepts": 51, "top": [[804, "car", 0.329], [774, "car", 0.203], [392, "car", 0.201]], "label_55": "car", "iou_55": 0.163, "counts_top": ["car", 41]}, "mixed5b": {"channels": 1024, "detectors": 277, "concepts": 51, "top": [[614, "car", 0.17], [0, "pool table", 0.162], [972, "pool table", 0.146]], "label_55": null, "iou_55": null, "counts_top": ["pool table", 55]}}, "street": {"ade": "ADE_val_00001696.jpg", "T": 257.64, "iou_photo": 0.777, "iou_all": 0.261}, "blocks": {"top5": [["beach wagon", 0.935], ["pickup", 0.033], ["cab", 0.013], ["minivan", 0.007], ["fire engine", 0.004]], "explained": 0.56, "attr": [[-2.094, 0.111, 0.393, -1.489, 1.691], [-0.107, 1.352, -1.149, 2.169, -0.522], [6.635, 1.738, 4.984, 1.687, -1.103], [1.026, 1.631, 0.425, 1.115, 4.351], [1.366, 1.51, 1.181, 0.541, 0.373], [4.249, 2.33, 2.33, 2.668, 0.232]], "names": ["car body", "wheels", "windows", "houses", "road", "rear"]}, "atlas": {"n_points": 100000, "grid": 32, "n_cells": 733, "top1": ["titi", "toy terrier", "patas", "titi", "Bouvier des Flandres", "Labrador retriever", "neck brace", "nipple", "Great Pyrenees", "kite", "grey fox", "otter", "American coot", "baboon", "guenon", "flute", "shower cap", "maillot tank suit", "sloth bear", "red-backed sandpiper", "dingo", "black-and-tan coonhound", "Afghan hound", "kite", "chickadee", "American Staffordshire terrier", "shower cap", "mortarboard", "assault rifle", "Scotch terrier", "proboscis monkey", "bighorn", "Japanese spaniel", "Brittany spaniel", "standard poodle", "black grouse", "redbone", "patas", "Great Dane", "neck brace", "stage", "bonnet", "black grouse", "Ibizan hound", "polecat", "badger", "mink", "weasel", "ptarmigan", "otterhound", "English setter", "weasel", "guenon", "Windsor tie", "ski mask", "oxygen mask", "harmonica", "potter's wheel", "siamang", "dragonfly", "black grouse", "limpkin", "goose", "soft-coated wheaten terrier", "weasel", "Indian elephant", "black grouse", "Bouvier des Flandres", "patas", "briard", "Tibetan terrier", "cello", "Windsor tie", "bathing cap", "miniskirt", "maillot tank suit", "balance beam", "cradle", "partridge", "house finch", "sea lion", "acorn", "crane bird", "jay", "black stork", "Arabian camel", "hartebeest", "ox", "Norwegian elkhound", "sloth bear", "Newfoundland", "wire-haired fox terrier", "Shih-Tzu", "football helmet", "stretcher", "potter's wheel", "maillot tank suit", "flute", "black stork", "plunger", "bubble", "ladybug", "trifle", "snail", "goldfinch", "jaguar", "magpie", "little blue heron", "brambling", "robin", "bluetick", "standard poodle", "Scottish deerhound", "Norfolk terrier", "miniature schnauzer", "bassoon", "stretcher", "ice lolly", "maraca", "tailed frog", "lorikeet", "American egret", "orange", "earthstar", "potpie", "meat loaf", "loggerhead", "leatherback turtle", "sulphur-crested cockatoo", "American egret", "European gallinule", "junco", "Indian elephant", "standard poodle", "Welsh springer spaniel", "standard poodle", "chain saw", "crash helmet", "sleeping bag", "bassinet", "missile", "eft", "tennis ball", "lemon", "carbonara", "head cabbage", "king crab", "hermit crab", "tailed frog", "terrapin", "house finch", "dowitcher", "house finch", "power drill", "Rottweiler", "Mexican hairless", "Lakeland terrier", "chain saw", "horse cart", "overskirt", "lab coat", "quill", "swab", "greenhouse", "eggnog", "carbonara", "coral fungus", "hen-of-the-woods", "sidewinder", "tench", "banded gecko", "black widow", "water ouzel", "crane bird", "killer whale", "junco", "Scottish deerhound", "wire-haired fox terrier", "espresso", "racket", "horse cart", "whippet", "paddle", "syringe", "bassinet", "golf ball", "honeycomb", "trilobite", "zebra", "head cabbage", "tiger beetle", "whiptail", "banded gecko", "wolf spider", "cardoon", "bittern", "goldfish", "eft", "mink", "Appenzeller", "reflex camera", "Petri dish", "consomme", "croquet ball", "pencil sharpener", "toucan", "bubble", "caldron", "mortar", "cheeseburger", "chain mail", "armadillo", "wool", "armadillo", "cabbage butterfly", "head cabbage", "alligator lizard", "wolf spider", "cardoon", "cockroach", "hammerhead", "kite", "white stork", "black stork", "magnetic compass", "loupe", "ski mask", "ladle", "airliner", "potter's wheel", "crash helmet", "eggnog", "balloon", "spaghetti squash", "dome", "wool", "rock beauty", "peacock", "cabbage butterfly", "hammer", "lacewing", "tick", "little blue heron", "goose", "airliner", "ski", "white stork", "European gallinule", "sloth bear", "pickup", "car wheel", "pencil sharpener", "toucan", "spotlight", "chain", "cup", "bucket", "mixing bowl", "mixing bowl", "pick", "dumbbell", "bolo tie", "chain", "electric ray", "knot", "hare", "oboe", "American egret", "European gallinule", "American chameleon", "weevil", "African grey", "weasel", "crane bird", "crane bird", "dumbbell", "crash helmet", "wall clock", "buckle", "holster", "washbasin", "water jug", "cocktail shaker", "barrel", "water jug", "stinkhorn", "measuring cup", "barracouta", "balance beam", "whistle", "rubber eraser", "hatchet", "plunger", "black stork", "hummingbird", "ringlet", "isopod", "tiger beetle", "common newt", "frilled lizard", "ground beetle", "dowitcher", "wild boar", "mink", "football helmet", "handkerchief", "safe", "ballpoint", "clog", "binoculars", "vestment", "soap dispenser", "hourglass", "vase", "breastplate", "hourglass", "jigsaw puzzle", "pick", "red-breasted merganser", "lumbermill", "matchstick", "horizontal bar", "safety pin", "walking stick", "garden spider", "tiger beetle", "head cabbage", "ringlet", "ground beetle", "jay", "isopod", "cricket", "toy poodle", "digital watch", "pop bottle", "sunscreen", "football helmet", "combination lock", "steel drum", "rifle", "parking meter", "swab", "tripod", "hourglass", "beaker", "cannon", "wooden spoon", "measuring cup", "reel", "bonnet", "goose", "pedestal", "Siamese cat", "redshank", "jay", "bee eater", "jackfruit", "head cabbage", "pot", "black-and-tan coonhound", "ringneck snake", "goldfish", "warthog", "tusker", "book jacket", "jigsaw puzzle", "comic book", "typewriter keyboard", "switch", "carpenter's kit", "cocktail shaker", "switch", "beacon", "jinrikisha", "plunger", "ladle", "flagpole", "broom", "overskirt", "potter's wheel", "holster", "knee pad", "submarine", "bow", "white wolf", "ground beetle", "cockroach", "garden spider", "damselfly", "damselfly", "leaf beetle", "rock crab", "coral reef", "wild boar", "mink", "crane bird", "brass", "packet", "space bar", "microwave", "carpenter's kit", "packet", "throne", "ashcan", "notebook", "assault rifle", "gondola", "chime", "hammer", "yawl", "harp", "water jug", "sax", "measuring cup", "paintbrush", "leaf beetle", "kelpie", "vulture", "black stork", "black and gold garden spider", "marmoset", "bittern", "earthstar", "common newt", "sea slug", "eel", "stinkhorn", "prairie chicken", "rule", "oscilloscope", "cassette", "minibus", "beach wagon", "bubble", "tow truck", "submarine", "rocking chair", "drilling platform", "paddlewheel", "turnstile", "plunger", "harp", "Windsor tie", "torch", "saltshaker", "milk can", "redbone", "bassoon", "malamute", "dhole", "prairie chicken", "coucal", "jacamar", "bee eater", "soccer ball", "rain barrel", "sea slug", "hognose snake", "mongoose", "prairie chicken", "cassette", "tape player", "printer", "moving van", "recreational vehicle", "tobacco shop", "tobacco shop", "organ", "drilling platform", "throne", "bookcase", "schooner", "crane", "medicine chest", "medicine chest", "rifle", "water tower", "ping-pong ball", "screw", "trombone", "tailed frog", "green mamba", "bustard", "frilled lizard", "bee eater", "komondor", "golf ball", "tailed frog", "chiton", "stinkhorn", "white stork", "thresher", "swab", "jeep", "garbage truck", "library", "bookcase", "boathouse", "palace", "monastery", "dock", "schooner", "pirate", "theater curtain", "shower curtain", "wardrobe", "broom", "reel", "planetarium", "airliner", "solar dish", "gar", "barracouta", "Maltese dog", "croquet ball", "kite", "dhole", "titi", "cliff", "cliff", "dhole", "prairie chicken", "thresher", "convertible", "tape player", "palace", "palace", "palace", "palace", "pier", "solar dish", "window screen", "radiator", "theater curtain", "broom", "broom", "plane", "candle", "shoji", "whistle", "spotlight", "Arabian camel", "black grouse", "black-and-tan coonhound", "barn spider", "African hunting dog", "American black bear", "coucal", "coucal", "alligator lizard", "mushroom", "bustard", "maze", "police van", "passenger car", "airliner", "airliner", "steel arch bridge", "upright", "lumbermill", "shoji", "shoji", "solar dish", "window screen", "microwave", "wardrobe", "barber chair", "consomme", "bow", "whistle", "common newt", "whistle", "tiger shark", "African chameleon", "horned viper", "barn spider", "hen-of-the-woods", "coral reef", "coral reef", "earthstar", "weasel", "vulture", "rapeseed", "lifeboat", "catamaran", "screen", "fireboat", "boathouse", "marimba", "printer", "window shade", "window shade", "window screen", "fire screen", "lakeside", "paper towel", "plunger", "whistle", "vase", "soap dispenser", "barn spider", "loggerhead", "catamaran", "bee eater", "coucal", "common newt", "common newt", "coral reef", "plow", "valley", "lakeside", "valley", "rule", "passenger car", "matchstick", "printer", "grille", "balance beam", "photocopier", "marimba", "bathtub", "plow", "potter's wheel", "can opener", "envelope", "great white shark", "jacamar", "red-breasted merganser", "lighter", "dragonfly", "fly", "coho", "leatherback turtle", "water snake", "common newt", "coyote", "hartebeest", "worm fence", "lakeside", "maze", "zucchini", "canoe", "hatchet", "electric locomotive", "croquet ball", "killer whale", "nail", "Australian terrier", "alligator lizard", "studio couch", "barn spider", "barn spider", "hook", "projector", "African chameleon", "nipple", "tick", "tick", "black grouse", "bathtub", "ram", "lakeside", "lakeside", "lakeside", "radio", "passenger car", "passenger car", "amphibian", "desk", "studio couch", "volleyball", "ashcan", "rubber eraser", "hair slide", "great white shark", "hair slide", "hook", "nematode", "lakeside", "sports car", "promontory", "black stork", "African crocodile", "geyser", "lakeside", "passenger car", "catamaran", "passenger car", "submarine", "studio couch", "sports car", "pool table", "speedboat", "hammerhead", "dugong", "amphibian", "dugong", "tiger shark", "tiger shark", "solar dish", "minibus", "valley", "promontory", "bicycle-built-for-two", "ski", "dogsled", "amphibian", "puck", "tiger shark", "dugong", "promontory", "dam", "catamaran", "red-breasted merganser", "submarine", "torch", "planetarium", "drilling platform", "lakeside", "volcano", "lakeside", "lakeside", "seashore", "grey whale", "seashore", "killer whale", "geyser", "seashore", "seashore", "valley", "geyser", "geyser", "seashore"]}, "whale": {"n_vectors": 20000, "n_cells": 173, "share_top2": 0.45, "top_cell": {"i": 0, "j": 3, "n": 131, "attr_whale": 0.044, "attr_shark": 0.502, "frac_from_shark_photos": 0.985, "baseball_hits": 498, "baseball_mean_cos": 0.464}}, "patch": {"n_whales": 1200, "n_ok": 1110, "clean_p_shark": 0.0026, "example": {"source": 0, "corner": 1, "size": 32, "p_before": [["grey whale", 0.387], ["great white shark", 0.206], ["tiger shark", 0.116]], "p_after": [["great white shark", 0.306], ["dugong", 0.249], ["grey whale", 0.228]]}, "runs": {"baseball|32": {"to_shark": 0.0012, "to_shark_best_case_any": 0.0054, "to_patch_class": 0.0011, "still_whale": 0.9628, "mean_dlog_p_shark": 1.108}, "baseball|48": {"to_shark": 0.0026, "to_shark_best_case_any": 0.0216, "to_patch_class": 0.0158, "still_whale": 0.887, "mean_dlog_p_shark": 1.929}, "baseball|64": {"to_shark": 0.0053, "to_shark_best_case_any": 0.045, "to_patch_class": 0.0435, "still_whale": 0.8266, "mean_dlog_p_shark": 2.694}, "baseball|80": {"to_shark": 0.0114, "to_shark_best_case_any": 0.0874, "to_patch_class": 0.0965, "still_whale": 0.6869, "mean_dlog_p_shark": 3.411}, "baseball|96": {"to_shark": 0.0304, "to_shark_best_case_any": 0.2117, "to_patch_class": 0.2516, "still_whale": 0.428, "mean_dlog_p_shark": 4.163}, "baseball|128": {"to_shark": 0.0541, "to_shark_best_case_any": 0.4162, "to_patch_class": 0.6942, "still_whale": 0.1559, "mean_dlog_p_shark": 3.801}, "tennis ball|32": {"to_shark": 0.0024, "to_shark_best_case_any": 0.009, "to_patch_class": 0.0153, "still_whale": 0.9053, "mean_dlog_p_shark": 1.74}, "tennis ball|48": {"to_shark": 0.0014, "to_shark_best_case_any": 0.0108, "to_patch_class": 0.0995, "still_whale": 0.7878, "mean_dlog_p_shark": 1.833}, "tennis ball|64": {"to_shark": 0.0014, "to_shark_best_case_any": 0.0117, "to_patch_class": 0.2524, "still_whale": 0.6012, "mean_dlog_p_shark": 1.72}, "tennis ball|80": {"to_shark": 0.0012, "to_shark_best_case_any": 0.0108, "to_patch_class": 0.3948, "still_whale": 0.4758, "mean_dlog_p_shark": 1.545}, "tennis ball|96": {"to_shark": 0.0005, "to_shark_best_case_any": 0.0072, "to_patch_class": 0.5655, "still_whale": 0.3296, "mean_dlog_p_shark": 0.705}, "tennis ball|128": {"to_shark": 0.0002, "to_shark_best_case_any": 0.0036, "to_patch_class": 0.8264, "still_whale": 0.1097, "mean_dlog_p_shark": -1.691}, "golf ball|32": {"to_shark": 0.0023, "to_shark_best_case_any": 0.009, "to_patch_class": 0.0001, "still_whale": 0.9566, "mean_dlog_p_shark": 1.17}, "golf ball|48": {"to_shark": 0.0015, "to_shark_best_case_any": 0.009, "to_patch_class": 0.0167, "still_whale": 0.8786, "mean_dlog_p_shark": 1.575}, "golf ball|64": {"to_shark": 0.001, "to_shark_best_case_any": 0.0072, "to_patch_class": 0.029, "still_whale": 0.8234, "mean_dlog_p_shark": 1.765}, "golf ball|80": {"to_shark": 0.0017, "to_shark_best_case_any": 0.018, "to_patch_class": 0.0384, "still_whale": 0.7903, "mean_dlog_p_shark": 2.282}, "golf ball|96": {"to_shark": 0.0028, "to_shark_best_case_any": 0.0279, "to_patch_class": 0.1466, "still_whale": 0.6589, "mean_dlog_p_shark": 2.36}, "golf ball|128": {"to_shark": 0.0236, "to_shark_best_case_any": 0.2505, "to_patch_class": 0.386, "still_whale": 0.4155, "mean_dlog_p_shark": 2.19}, "soccer ball|32": {"to_shark": 0.0009, "to_shark_best_case_any": 0.0081, "to_patch_class": 0.0003, "still_whale": 0.9731, "mean_dlog_p_shark": 0.687}, "soccer ball|48": {"to_shark": 0.0015, "to_shark_best_case_any": 0.0126, "to_patch_class": 0.0059, "still_whale": 0.8985, "mean_dlog_p_shark": 1.609}, "soccer ball|64": {"to_shark": 0.0034, "to_shark_best_case_any": 0.0333, "to_patch_class": 0.0475, "still_whale": 0.7232, "mean_dlog_p_shark": 2.488}, "soccer ball|80": {"to_shark": 0.003, "to_shark_best_case_any": 0.0315, "to_patch_class": 0.0555, "still_whale": 0.6342, "mean_dlog_p_shark": 2.605}, "soccer ball|96": {"to_shark": 0.0017, "to_shark_best_case_any": 0.0189, "to_patch_class": 0.1191, "still_whale": 0.5046, "mean_dlog_p_shark": 2.278}, "soccer ball|128": {"to_shark": 0.0008, "to_shark_best_case_any": 0.0108, "to_patch_class": 0.316, "still_whale": 0.2855, "mean_dlog_p_shark": 1.579}, "orange|32": {"to_shark": 0.0023, "to_shark_best_case_any": 0.0054, "to_patch_class": 0.0, "still_whale": 0.9422, "mean_dlog_p_shark": 1.152}, "orange|48": {"to_shark": 0.0003, "to_shark_best_case_any": 0.0018, "to_patch_class": 0.0, "still_whale": 0.9213, "mean_dlog_p_shark": 1.186}, "orange|64": {"to_shark": 0.0003, "to_shark_best_case_any": 0.0027, "to_patch_class": 0.0015, "still_whale": 0.8893, "mean_dlog_p_shark": 1.452}, "orange|80": {"to_shark": 0.0003, "to_shark_best_case_any": 0.0018, "to_patch_class": 0.0163, "still_whale": 0.8577, "mean_dlog_p_shark": 1.662}, "orange|96": {"to_shark": 0.0006, "to_shark_best_case_any": 0.0063, "to_patch_class": 0.0647, "still_whale": 0.8062, "mean_dlog_p_shark": 1.666}, "orange|128": {"to_shark": 0.0007, "to_shark_best_case_any": 0.0045, "to_patch_class": 0.3351, "still_whale": 0.5615, "mean_dlog_p_shark": 1.586}, "noise|32": {"to_shark": 0.0, "to_shark_best_case_any": 0.0, "to_patch_class": 0.0, "still_whale": 0.9886, "mean_dlog_p_shark": 0.249}, "noise|48": {"to_shark": 0.0005, "to_shark_best_case_any": 0.0018, "to_patch_class": 0.0, "still_whale": 0.9714, "mean_dlog_p_shark": 0.597}, "noise|64": {"to_shark": 0.0009, "to_shark_best_case_any": 0.0027, "to_patch_class": 0.0, "still_whale": 0.9538, "mean_dlog_p_shark": 0.705}, "noise|80": {"to_shark": 0.001, "to_shark_best_case_any": 0.0045, "to_patch_class": 0.0, "still_whale": 0.9318, "mean_dlog_p_shark": 1.305}, "noise|96": {"to_shark": 0.0017, "to_shark_best_case_any": 0.0072, "to_patch_class": 0.0, "still_whale": 0.8832, "mean_dlog_p_shark": 1.908}, "noise|128": {"to_shark": 0.0027, "to_shark_best_case_any": 0.0144, "to_patch_class": 0.0, "still_whale": 0.6819, "mean_dlog_p_shark": 2.857}, "grey|32": {"to_shark": 0.0027, "to_shark_best_case_any": 0.0045, "to_patch_class": 0.0, "still_whale": 0.948, "mean_dlog_p_shark": 1.171}, "grey|48": {"to_shark": 0.0014, "to_shark_best_case_any": 0.0036, "to_patch_class": 0.0, "still_whale": 0.9011, "mean_dlog_p_shark": 1.567}, "grey|64": {"to_shark": 0.0014, "to_shark_best_case_any": 0.0045, "to_patch_class": 0.0, "still_whale": 0.8628, "mean_dlog_p_shark": 1.78}, "grey|80": {"to_shark": 0.0016, "to_shark_best_case_any": 0.0054, "to_patch_class": 0.0, "still_whale": 0.8164, "mean_dlog_p_shark": 2.211}, "grey|96": {"to_shark": 0.0036, "to_shark_best_case_any": 0.0117, "to_patch_class": 0.0, "still_whale": 0.7259, "mean_dlog_p_shark": 2.609}, "grey|128": {"to_shark": 0.0056, "to_shark_best_case_any": 0.0162, "to_patch_class": 0.0, "still_whale": 0.4966, "mean_dlog_p_shark": 3.078}, "baseball_on_other_photos|64": {"to_shark": 0.0005}, "baseball_on_other_photos|96": {"to_shark": 0.0005}}, "sources": {"baseball": ["images/429/val_36972.jpeg", "images/429/val_41253.jpeg", "images/429/val_22765.jpeg", "images/429/val_13260.jpeg", "images/429/val_09183.jpeg"], "tennis ball": ["images/852/val_35187.jpeg", "images/852/val_12658.jpeg", "images/852/val_25080.jpeg", "images/852/val_27390.jpeg", "images/852/val_20000.jpeg"], "golf ball": ["images/574/val_03792.jpeg", "images/574/val_48516.jpeg", "images/574/val_38789.jpeg", "images/574/val_36170.jpeg", "images/574/val_35917.jpeg"], "soccer ball": ["images/805/val_09300.jpeg", "images/805/val_47886.jpeg", "images/805/val_37178.jpeg", "images/805/val_13364.jpeg", "images/805/val_09802.jpeg"], "orange": ["images/950/val_03618.jpeg", "images/950/val_27082.jpeg", "images/950/val_14310.jpeg", "images/950/val_06301.jpeg", "images/950/val_27519.jpeg"]}}}''')
print(len(LECTURE['atlas']['top1']), 'cells in the lecture\'s atlas')

In [ ]:
# The photos: ImageNet's 50,000 validation photos, in the exact copy the course measured (a public 256-pixel copy,
# benjamin-paine/imagenet-1k-256x256 on Hugging Face: two parquet files of 25,000 photos, about 350 MB each, no key
# needed). Photo n here is photo n in the lecture. We read 100 photos at a time, only the ones we ask for.
import pyarrow.parquet as pq
import requests
from PIL import Image

SHARD = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "validation-{:05d}-of-00002.parquet")
_files = {}


def _shard(s):
    path = f"val-{s}.parquet"
    if not os.path.exists(path):
        t = time.time()
        with requests.get(SHARD.format(s), stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded photo file {s} ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t:.0f} s")
    if s not in _files:
        _files[s] = pq.ParquetFile(path)
    return _files[s]


def rows(ids):
    """{n: (jpeg bytes, class number 0..999)} for the photos with these numbers."""
    groups = {}
    for n in ids:
        groups.setdefault((n // 25000, (n % 25000) // 100), []).append(int(n))
    out = {}
    for (s, g), ns in sorted(groups.items()):
        t = _shard(s).read_row_group(g, columns=["image", "label"])
        ims, labs = t.column("image").to_pylist(), t.column("label").to_pylist()
        for n in ns:
            out[n] = (ims[n % 100]["bytes"], int(labs[n % 100]))
    return out


def decode(b, size=224):
    """JPEG bytes (256 px square) -> the centre 224 x 224, as a 3 x 224 x 224 tensor in [0, 1]."""
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    if (w, h) != (256, 256):
        s = 256 / min(w, h)
        im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
        w, h = im.size
    l, t = (w - size) // 2, (h - size) // 2
    im = im.crop((l, t, l + size, t + size))
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)

In [ ]:
# InceptionV1 (GoogLeNet, 2014), the network this lecture studies, through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()

import contextlib
from lucent.modelzoo import inceptionv1
with contextlib.redirect_stderr(io.StringIO()):            # keep the download's percentage bar off the page
    net = inceptionv1(pretrained=True).eval().to(DEV)
print("InceptionV1's weights (inception5h, 27 MB): ready")
for p in net.parameters():
    p.requires_grad_(False)

# Make sure the fix took: watch the normalization during one pass.
_M = importlib.import_module("lucent.modelzoo.inceptionv1.InceptionV1")
_seen, _orig = [], _M.F.local_response_norm
_M.F.local_response_norm = lambda x, size, alpha=1e-4, beta=0.75, k=1.0: (_seen.append((size, k)), _orig(x, size, alpha, beta, k))[1]
with torch.no_grad():
    net(torch.zeros(1, 3, 224, 224, device=DEV))
_M.F.local_response_norm = _orig
check("normalization size and k, every call", sorted(set(_seen)), [(10, 2)])


def prep(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


class Taps:
    """Look inside the network. record=[module names] keeps their outputs in taps[name];
    edit={name: fn} replaces a module's output with fn(output) on the way through."""
    def __init__(self, record=(), edit=None):
        mods = dict(net.named_modules())
        self.out, self.h = {}, []
        for name in record:
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, n=name: self.out.__setitem__(n, o)))
        for name, fn in (edit or {}).items():
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, fn=fn: fn(o)))

    def __getitem__(self, name):
        return self.out[name]

    def __enter__(self):
        return self

    def __exit__(self, *a):
        for h in self.h:
            h.remove()


# InceptionV1 has 1,008 outputs in its own order; PERM puts the 1,000 classes in the standard order (the lecture's
# perm.json, which also fixes two mislabelled outputs), and NAMES gives each class its name.
PERM = torch.tensor(json.loads("[449, 450, 442, 443, 444, 445, 446, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 494, 495, 496, 497, 498, 499, 500, 501, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 474, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 214, 215, 217, 216, 213, 218, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 226, 613, 614, 615, 616, 617, 618, 619, 620, 612, 422, 423, 424, 425, 427, 426, 428, 429, 430, 438, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 6, 22, 193, 14, 173, 99, 87, 69, 116, 198, 43, 89, 200, 98, 161, 132, 32, 180, 42, 18, 207, 181, 105, 77, 189, 20, 204, 63, 31, 66, 21, 25, 45, 170, 119, 210, 107, 126, 88, 145, 59, 160, 152, 72, 33, 91, 4, 27, 113, 123, 36, 156, 109, 158, 131, 26, 71, 56, 146, 144, 125, 176, 139, 134, 90, 2, 192, 154, 150, 94, 5, 19, 191, 196, 117, 141, 68, 93, 51, 208, 184, 97, 29, 171, 124, 128, 47, 64, 211, 112, 187, 114, 86, 115, 79, 130, 140, 110, 82, 17, 177, 149, 15, 3, 41, 127, 84, 143, 133, 60, 172, 179, 118, 168, 148, 70, 197, 50, 106, 49, 151, 46, 205, 102, 28, 58, 155, 136, 202, 34, 62, 1, 159, 67, 174, 55, 10, 95, 8, 11, 201, 85, 153, 30, 190, 76, 206, 61, 163, 103, 209, 74, 35, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 224, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 225, 657, 658, 188, 129, 164, 157, 13, 53, 183, 195, 101, 39, 80, 147, 78, 120, 167, 108, 162, 165, 81, 52, 9, 65, 57, 12, 121, 186, 48, 23, 182, 40, 212, 44, 16, 178, 38, 83, 104, 96, 185, 122, 73, 135, 137, 138, 203, 54, 100, 175, 142, 166, 37, 111, 92, 199, 75, 194, 24, 7, 169, 447, 451, 448, 452, 453, 456, 457, 454, 455, 547, 853, 896, 223, 345, 246, 230, 232, 677, 265, 264, 522, 688, 845, 752, 540, 847, 704, 767, 233, 907, 967, 341, 718, 916, 307, 705, 683, 518, 905, 258, 807, 908, 296, 353, 785, 909, 884, 266, 733, 991, 849, 777, 811, 933, 941, 254, 985, 835, 533, 839, 689, 252, 940, 805, 300, 706, 779, 538, 817, 716, 872, 719, 949, 851, 820, 580, 833, 887, 707, 267, 673, 591, 539, 239, 377, 836, 576, 568, 891, 749, 563, 548, 890, 929, 701, 241, 987, 342, 914, 827, 721, 902, 382, 762, 303, 335, 301, 801, 690, 695, 370, 713, 797, 979, 761, 996, 674, 698, 583, 543, 708, 243, 268, 376, 347, 910, 881, 297, 545, 778, 898, 298, 670, 756, 856, 865, 720, 313, 550, 959, 966, 523, 529, 315, 821, 667, 579, 715, 253, 897, 972, 834, 336, 799, 1000, 662, 512, 346, 262, 316, 879, 660, 808, 796, 305, 235, 279, 919, 995, 356, 309, 784, 261, 712, 934, 299, 256, 348, 671, 757, 280, 971, 567, 955, 275, 792, 276, 236, 337, 911, 227, 684, 725, 703, 517, 581, 895, 249, 375, 840, 505, 551, 750, 573, 350, 344, 554, 369, 787, 696, 730, 562, 802, 924, 293, 525, 980, 659, 592, 748, 269, 961, 963, 294, 560, 770, 773, 582, 918, 892, 814, 228, 374, 988, 371, 687, 238, 546, 270, 244, 867, 973, 894, 508, 536, 697, 532, 818, 917, 782, 977, 763, 338, 339, 781, 984, 598, 922, 946, 302, 717, 509, 661, 866, 875, 920, 880, 271, 251, 871, 829, 290, 272, 764, 686, 869, 277, 824, 854, 691, 852, 260, 255, 728, 511, 599, 284, 588, 585, 595, 755, 915, 552, 699, 354, 351, 520, 513, 333, 870, 937, 295, 506, 921, 826, 564, 584, 504, 759, 685, 352, 877, 942, 994, 306, 527, 257, 679, 843, 732, 842, 850, 883, 783, 789, 575, 926, 722, 281, 596, 931, 901, 888, 841, 565, 245, 983, 372, 693, 964, 731, 381, 378, 857, 923, 285, 855, 314, 788, 838, 566, 373, 769, 556, 702, 542, 534, 572, 846, 939, 862, 976, 273, 860, 571, 863, 537, 927, 286, 570, 965, 668, 578, 694, 219, 541, 310, 663, 997, 876, 519, 760, 753, 586, 952, 751, 938, 355, 809, 521, 962, 221, 729, 510, 587, 379, 589, 559, 800, 709, 832, 950, 259, 380, 868, 747, 590, 776, 943, 549, 726, 557, 507, 288, 278, 960, 222, 986, 577, 925, 822, 858, 515, 234, 676, 237, 600, 775, 274, 593, 804, 263, 680, 340, 530, 998, 724, 528, 516, 514, 287, 957, 311, 692, 247, 794, 526, 574, 535, 810, 681, 828, 837, 945, 569, 561, 531, 304, 250, 978, 675, 786, 944, 970, 989, 903, 758, 555, 308, 780, 664, 710, 312, 594, 700, 282, 711, 289, 283, 766, 825, 291, 242, 597, 678, 882, 349, 765, 727, 544, 220, 292, 334, 666, 874, 990, 969, 558, 790, 682, 343, 936, 665, 524, 928, 317, 231, 906, 669, 958, 819, 795, 772, 502, 899, 912, 904, 935, 831, 503, 672, 951, 815, 723, 248, 240, 714, 553, 930, 791, 932, 861, 774, 803, 754, 813, 844, 771, 793, 974, 968, 873, 768, 975, 993, 885, 734, 736, 737, 738, 739, 740, 741, 742, 743, 744, 735, 745, 746, 318, 229, 319, 320, 321, 322, 323, 324, 325, 326, 816, 999, 953, 864, 806, 948, 830, 900, 798, 947, 859, 823, 361, 992, 359, 365, 368, 366, 363, 364, 367, 360, 362, 954, 848, 982, 330, 357, 358, 331, 327, 328, 332, 886, 913, 956, 893, 878, 812, 981, 329, 889]"), device=DEV)
from torchvision.models import GoogLeNet_Weights
NAMES = GoogLeNet_Weights.IMAGENET1K_V1.meta["categories"]

In [ ]:
import zipfile, torch.nn as nn
LAYERS = ["conv2d0", "conv2d1", "conv2d2", "mixed3a", "mixed3b", "mixed4a", "mixed4b", "mixed4c", "mixed4d",
          "mixed4e", "mixed5a", "mixed5b"]
for _m in net.modules():                         # keep lucent's drawing trick, but measure with the plain gradient
    if type(_m).__name__ == "RedirectedReluLayer":
        _m._redirect = _m.forward


def set_relu(mode):
    for m in net.modules():
        if hasattr(m, "_redirect"):
            m.forward = m._redirect if mode == "redirect" else (lambda x: F.relu(x))

set_relu("plain")
INKMAP = plt.matplotlib.colors.LinearSegmentedColormap.from_list("ink", [PAPER, INK])
WHALE, SHARK, BASEBALL = NAMES.index("grey whale"), NAMES.index("great white shark"), NAMES.index("baseball")


def forward_from(layer, x):
    """Run the network on x with `layer`'s output as the start of the gradient's road: the slopes we need (of the class
    scores with respect to that layer) come out exactly the same, and nothing below the layer is kept for them."""
    store = {}
    def leaf(m, i, o):
        store["h"] = o.detach().requires_grad_(True)
        return store["h"]
    hk = dict(net.named_modules())[layer].register_forward_hook(leaf)
    try:
        with Taps(["softmax2_pre_activation_matmul"]) as T:
            net(prep(x))
    finally:
        hk.remove()
    return store["h"], T["softmax2_pre_activation_matmul"][:, PERM]

## 1. Too many to draw (Plate I)
Lecture 8 drew six neurons carefully. How many are there in the twelve layers we use?

In [ ]:
with torch.no_grad(), Taps(LAYERS) as T:
    net(torch.zeros(1, 3, 224, 224, device=DEV))
total = sum(T[L].shape[1] for L in LAYERS)
check("channels in InceptionV1's 12 layers", total, LECTURE["channels_total"])

## 2. Labelled pixels, and IoU (Plates II–III)
**ADE20K** (Zhou and colleagues, 2017) is a set of photos in which people have named the thing under every pixel, from
150 kinds: a **segmentation mask**. We use its 2,000 test photos, each cut to its centre square: 224 × 224 for the
network, the mask at 56 × 56 (nearest label, no blending).

In [ ]:
ADE = "https://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip"
if not os.path.exists("ade.zip"):
    t = time.time()
    with requests.get(ADE, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open("ade.zip.part", "wb") as f:
            for chunk in r.iter_content(1 << 22):
                f.write(chunk)
    os.rename("ade.zip.part", "ade.zip")
    print(f"downloaded ADE20K ({os.path.getsize('ade.zip') / 1e6:.0f} MB) in {time.time() - t:.0f} s")
Z = zipfile.ZipFile("ade.zip")
info = Z.read("ADEChallengeData2016/objectInfo150.txt").decode().splitlines()[1:]
ADE_NAMES = ["other"] + [l.split("\t")[-1].split(";")[0].strip() for l in info]
ade_files = sorted(n for n in Z.namelist() if n.startswith("ADEChallengeData2016/images/validation/") and n.endswith(".jpg"))


def ade_load(n, R=56):
    im = Image.open(io.BytesIO(Z.read(n))).convert("RGB")
    mk = Image.open(io.BytesIO(Z.read(n.replace("/images/", "/annotations/").replace(".jpg", ".png"))))
    w, h = im.size
    s = min(w, h)
    box = ((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2)
    x = torch.from_numpy(np.asarray(im.crop(box).resize((224, 224), Image.BICUBIC), np.float32) / 255).permute(2, 0, 1)
    return x, torch.from_numpy(np.asarray(mk.crop(box).resize((R, R), Image.NEAREST), np.int64))


t = time.time()
ade = [ade_load(n) for n in ade_files]
AX = torch.stack([a[0] for a in ade]); AM = torch.stack([a[1] for a in ade])
del ade
print(f"{len(AX):,} labelled test photos ready in {time.time() - t:.0f} s")
check("ADE20K test photos", len(AX), 2000)
CAR = next(i for i, n in enumerate(ADE_NAMES) if n.split(",")[0].strip() == "car")

**Does a neuron fire where the cars are?** Take the car neuron, 4c:447. Upsample its map to 56 × 56, keep only the
squares where it beats its **threshold** (its top 0.5% of values, measured on the first 400 photos), and lay them on
the car pixels. Overlap ÷ union is the **IoU**: 1 is a perfect match, 0 no overlap.

In [ ]:
def maps(L, X, R=56):
    with torch.no_grad(), Taps([L]) as T:
        net(prep(X.to(DEV)))
    return F.interpolate(T[L], size=(R, R), mode="bilinear", align_corners=False)


def thresholds(L):
    """Each channel's top-0.5% value, from its upsampled maps on the first 400 photos (every 7th value, as the lecture)."""
    samp = torch.cat([maps(L, AX[i:i + 50]).permute(1, 0, 2, 3).flatten(1)[:, ::7].cpu() for i in range(0, 400, 50)], 1)
    return torch.quantile(samp.float(), 0.995, dim=1).to(DEV)


T4c = thresholds("mixed4c")
check("4c:447's threshold", T4c[447].item(), LECTURE["street"]["T"], tol=0.05)
k = next(i for i, n in enumerate(ade_files) if n.endswith(LECTURE["street"]["ade"]))
on = (maps("mixed4c", AX[k:k + 1])[0, 447] > T4c[447]).cpu()
car = AM[k] == CAR
iou_k = float((on & car).sum() / (on | car).sum())
check(f"IoU on {LECTURE['street']['ade']}", iou_k, LECTURE["street"]["iou_photo"], tol=0.002)
fig, axs = plt.subplots(1, 3, figsize=(8.4, 2.9))
axs[0].imshow(AX[k].permute(1, 2, 0)); axs[0].set_title("an ADE20K street", fontsize=9)
axs[1].imshow(car, cmap=INKMAP); axs[1].set_title("its car pixels", fontsize=9)
axs[2].imshow(car.float() * 0.35 + on.float() * 0.65, cmap=INKMAP); axs[2].set_title(f"4c:447 above threshold: IoU {iou_k:.3f}", fontsize=9)
for a in axs:
    a.axis("off")
plt.show()

## 3. Label every neuron (Plate IV)
**Network Dissection** (Bau and colleagues, 2017): for every channel of a layer and every one of the 150 labels, sum
the overlap and the union over all 2,000 photos, and keep the best label. A channel whose best IoU beats 0.04 is a
**detector**. Four layers, from the middle of the network to the end.

In [ ]:
def dissect(L):
    T = T4c if L == "mixed4c" else thresholds(L)
    C = T.numel()
    inter = torch.zeros(C, 151, device=DEV)
    area_a, area_l = torch.zeros(C, device=DEV), torch.zeros(151, device=DEV)
    for i in range(0, len(AX), 50):
        a = (maps(L, AX[i:i + 50]) > T[None, :, None, None]).float()
        oh = F.one_hot(AM[i:i + 50].to(DEV), 151).permute(0, 3, 1, 2).float()
        inter += torch.einsum("bchw,blhw->cl", a, oh)
        area_a += a.sum((0, 2, 3)); area_l += oh.sum((0, 2, 3))
    iou = inter / (area_a[:, None] + area_l[None, :] - inter + 1e-6)
    iou[:, 0] = 0                                                # "other" is not a concept
    best = iou.max(1)
    return best.indices.cpu().numpy(), best.values.cpu().numpy(), iou


t = time.time()
DISSECT = {}
for L in ("mixed3b", "mixed4c", "mixed4e", "mixed5b"):
    lab, val, iou = dissect(L)
    det = val > 0.04
    concepts = sorted(set(ADE_NAMES[c] for c in lab[det]))
    DISSECT[L] = (lab, val)
    from collections import Counter
    common = Counter(ADE_NAMES[c].split(",")[0] for c in lab[det]).most_common(4)
    print(f"{L}: {det.sum()} detectors of {len(val)}, {len(concepts)} concepts; most common: {common}")
    want = LECTURE["dissect"][L]
    check(f"{L}: detectors", int(det.sum()), want["detectors"], tol=2)
    check(f"{L}: distinct concepts", len(concepts), want["concepts"], tol=1)
    check(f"{L}: its most common label", list(common[0]), list(want["counts_top"]), )
    c0 = int(np.argmax(val))
    check(f"{L}: its best detector", [c0, ADE_NAMES[lab[c0]].split(",")[0]], want["top"][0][:2])
    check(f"{L}: its best IoU", float(val[c0]), want["top"][0][2], tol=0.002)
print(f"dissected four layers in {time.time() - t:.0f} s")
lab4c, val4c = DISSECT["mixed4c"]
check("4c:447, found blind, is labelled", ADE_NAMES[lab4c[447]].split(",")[0], "car")
check("4c:447's IoU with car over all 2,000 photos", float(val4c[447]), LECTURE["street"]["iou_all"], tol=0.001)

fig, ax = plt.subplots(figsize=(6, 2.6))
Ls = list(DISSECT)
ax.bar(range(4), [(DISSECT[L][1] > 0.04).mean() for L in Ls], color=PAPER, edgecolor=INK, hatch="///")
ax.set_xticks(range(4)); ax.set_xticklabels(Ls); ax.set_ylabel("share that are detectors"); ax.set_ylim(0, 0.5)
plt.show()

Dissection only finds what its dataset names. ADE20K labels streets and rooms, with cars and almost no cats, so the
"cat / car" neuron 4e:55 of Lecture 6 comes out a car detector here.

In [ ]:
lab4e, val4e = DISSECT["mixed4e"]
print(f"4e:55: labelled {ADE_NAMES[lab4e[55]].split(',')[0]}, IoU {val4e[55]:.3f}")
check("4e:55's label", ADE_NAMES[lab4e[55]].split(",")[0], LECTURE["dissect"]["mixed4e"]["label_55"])
check("4e:55's IoU", float(val4e[55]), LECTURE["dissect"]["mixed4e"]["iou_55"], tol=0.002)
del AX, AM

## 4. One photo, taken apart (Plates V–VI)
**Building blocks** (Olah and colleagues, 2018). Lecture 6's cab, at mixed4d: at each of its 14 × 14 = 196 spots, 528
numbers. Find 6 recurring mixes of channels that rebuild those numbers (**non-negative matrix factorisation**: numbers
≥ 0 written as 6 maps × 6 mixes). Then **attribution**: how hard each group pushed the network toward each of its top
five answers (Lecture 7's slope times how much of the group is present).

In [ ]:
from sklearn.decomposition import NMF
cab = decode(rows([3111])[3111][0])
x = cab[None].to(DEV).requires_grad_(True)
with Taps(["mixed4d", "softmax2_pre_activation_matmul"]) as T:
    net(prep(x))
A4d = T["mixed4d"]
lg = T["softmax2_pre_activation_matmul"][:, PERM][0]
top = lg.softmax(0).topk(5)
for (name, p), v, c in zip(LECTURE["blocks"]["top5"], top.values, top.indices):
    check(f"the cab: {name}", [NAMES[int(c)], round(float(v), 3)], [name, p])
C, H, W = A4d.shape[1:]
acts = np.ascontiguousarray(A4d[0].detach().permute(1, 2, 0).reshape(H * W, C).cpu().numpy())   # (196 spots, 528 numbers)
import warnings
from sklearn.decomposition._nmf import _initialize_nmf
# The lecture's settings: NNDSVD start, 1,000 steps, seed 0. We compute that same start with scikit-learn's own
# function and hand it over in row order (its solver rejects the column-ordered copy it would otherwise make here).
W0, H0 = _initialize_nmf(acts, 6, init="nndsvd", random_state=0)
nmf = NMF(n_components=6, init="custom", max_iter=1000, random_state=0)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")        # a possible "not fully converged" notice
    S = nmf.fit_transform(acts, W=np.ascontiguousarray(W0), H=np.ascontiguousarray(H0))   # (196, 6): where each group is
D = nmf.components_                        # (6, 528): what each group is
explained = 1 - np.linalg.norm(acts - S @ D) / np.linalg.norm(acts)
check("six groups rebuild this share of the numbers", explained, LECTURE["blocks"]["explained"], tol=0.005)
attr = np.zeros((6, 5))
for j, c in enumerate(top.indices.tolist()):
    g, = torch.autograd.grad(lg[c], A4d, retain_graph=True)
    G = g[0].permute(1, 2, 0).reshape(H * W, C).detach().cpu().numpy()
    for f in range(6):
        attr[f, j] = (np.outer(S[:, f], D[f]) * G).sum()
for f, name in enumerate(LECTURE["blocks"]["names"]):
    print(f"{name:9s}  toward beach wagon {attr[f, 0]:+.2f}")
    # the factorisation lands within about 1% of the lecture's on newer numerical libraries, hence the margin
    check(f"{name}: attribution to beach wagon", attr[f, 0], LECTURE["blocks"]["attr"][f][0], tol=0.1)
check("the group that made it a station wagon", LECTURE["blocks"]["names"][int(attr[:, 0].argmax())], "windows")

**Draw each group** by Lecture 8's method, with one change: instead of one neuron, the objective is a **direction**,
a mix of all 528 channels (how far each spot's numbers point along the group's mix, weighted by how closely they line up).
The same few lines draw the atlas cells below.

In [ ]:
import math
COLOR = np.array([[0.26, 0.09, 0.02], [0.27, 0.00, -0.05], [0.27, -0.09, 0.03]], np.float32)
COLOR = COLOR / np.max(np.linalg.norm(COLOR, axis=0))


def jitter(x, d):
    dx, dy = np.random.randint(d), np.random.randint(d)
    return x[:, :, dy:x.shape[2] - d + dy, dx:x.shape[3] - d + dx]


def standard(x):                                    # Lucid 2017's transforms (Lecture 8)
    x = jitter(F.pad(x, (12, 12, 12, 12), value=0.5), 8)
    s = float(np.random.choice([1 + (i - 5) / 50.0 for i in range(11)]))
    x = F.interpolate(x, scale_factor=s, mode="bilinear", align_corners=False) if s != 1 else x
    a = math.radians(float(np.random.choice(list(range(-10, 11)) + 5 * [0])))
    th = torch.tensor([[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0]], dtype=x.dtype, device=x.device)
    x = F.grid_sample(x, F.affine_grid(th[None].expand(x.shape[0], 2, 3), x.shape, align_corners=False),
                      mode="bilinear", padding_mode="reflection", align_corners=False)
    return jitter(x, 4)


class _Stop(Exception):
    pass


def capture(layer, x):
    out = {}
    def hook(m, i, o):
        out["a"] = o
        raise _Stop
    h = dict(net.named_modules())[layer].register_forward_hook(hook)
    try:
        net(prep(x))
    except _Stop:
        pass
    finally:
        h.remove()
    return out["a"]


def draw_directions(layer, V, size=128, steps=512, seed=0):
    """One picture per direction V[b] (B, C): Fourier parameterisation, Lucid 2017 transforms, dot x cosine objective."""
    np.random.seed(seed); torch.manual_seed(seed)
    set_relu("redirect")
    B, g = len(V), torch.Generator().manual_seed(seed)
    fy, fx = np.fft.fftfreq(size)[:, None], np.fft.fftfreq(size)[: size // 2 + 1]
    freq = np.sqrt(fx * fx + fy * fy)
    t = (torch.randn(B, 3, *freq.shape, 2, generator=g) * 0.01).to(DEV).requires_grad_(True)
    scale = torch.tensor(1.0 / np.maximum(freq, 1.0 / size), dtype=torch.float32, device=DEV)[None, None, ..., None]
    Cm, V = torch.tensor(COLOR, device=DEV), torch.as_tensor(V, dtype=torch.float32, device=DEV)
    def image():
        s = scale * t
        x = torch.fft.irfftn(torch.view_as_complex(s.contiguous()), s=(size, size), norm="ortho") / 4.0
        return torch.sigmoid(torch.einsum("bchw,dc->bdhw", x, Cm))
    opt = torch.optim.Adam([t], lr=0.05)
    for _ in range(steps):
        a = capture(layer, standard(image()))
        dot = (a * V[:, :, None, None]).sum(1)
        cos = dot / (a.norm(dim=1) * V.norm(dim=1)[:, None, None] + 1e-6)
        obj = (dot * cos.clamp(min=0.1)).mean((1, 2))
        opt.zero_grad(); (-obj.sum()).backward(); opt.step()
    set_relu("plain")
    return image().detach().cpu()


groups = draw_directions("mixed4d", D)
fig, axs = plt.subplots(2, 7, figsize=(11, 3.6))
axs[0, 0].imshow(cab.permute(1, 2, 0)); axs[0, 0].set_title("the cab", fontsize=9)
axs[1, 0].axis("off")
for f, name in enumerate(LECTURE["blocks"]["names"]):
    axs[0, f + 1].imshow(groups[f].permute(1, 2, 0)); axs[0, f + 1].set_title(name, fontsize=9)
    axs[1, f + 1].imshow(S[:, f].reshape(14, 14), cmap=INKMAP); axs[1, f + 1].set_title(f"{attr[f, 0]:+.1f}", fontsize=9)
for a in axs.flat:
    a.axis("off")
fig.suptitle("six building blocks of the cab: each group drawn, where it sits, and its push toward 'beach wagon'", fontsize=10)
plt.show()

## 5. Every activation, one map (Plates VII–VIII)
**Activation Atlas** (Carter and colleagues, 2019). At one spot of one photo, mixed4c holds 512 numbers: an
**activation vector**. Take many of them from many photos; give each a label by **attribution** (how hard it pushes
toward each of its photo's top five answers); place them on paper so that similar ones sit close (**UMAP**, after
**PCA** keeps the 50 most important directions of the vectors); lay a grid over the map; draw each cell's average.

The lecture used 4 random spots in each of its 50,000 test photos, 100,000 vectors, a 32 × 32 grid. To stay inside free
Colab's time we take every tenth photo (5,000 photos), the same 4-spots recipe (20,000 vectors) and a 16 × 16 grid.

In [ ]:
ids = list(range(0, 50000, 10))
rng = np.random.default_rng(0)
pos = rng.integers(1, 13, size=(len(ids), 4, 2))         # 4 random interior spots of the 14 x 14 grid, per photo
vec, att, cls5 = [], [], []
t = time.time()
for b0 in range(0, len(ids), 100):
    got = rows(ids[b0:b0 + 100])
    x = torch.stack([decode(got[i][0]) for i in ids[b0:b0 + 100]]).to(DEV)
    h, lg = forward_from("mixed4c", x)
    c5 = lg.topk(5, 1).indices
    bi = torch.arange(len(x), device=DEV)
    p = torch.from_numpy(pos[b0:b0 + 100]).to(DEV)
    a5 = []
    for kk in range(5):
        g, = torch.autograd.grad(lg[bi, c5[:, kk]].sum(), h, retain_graph=kk < 4)
        a5.append((h[bi[:, None], :, p[..., 0], p[..., 1]] * g[bi[:, None], :, p[..., 0], p[..., 1]]).sum(-1))
    vec.append(h[bi[:, None], :, p[..., 0], p[..., 1]].detach().reshape(-1, 512).cpu())
    att.append(torch.stack(a5, -1).detach().reshape(-1, 5).cpu())
    cls5.append(c5.repeat_interleave(4, 0).cpu())
X = torch.cat(vec).numpy(); Att = torch.cat(att).numpy(); Cls = torch.cat(cls5).numpy()
print(f"{len(X):,} activation vectors with their attributions in {time.time() - t:.0f} s")

In [ ]:
import warnings
from sklearn.decomposition import PCA
with warnings.catch_warnings():
    warnings.simplefilter("ignore")                         # umap's notices about threads and seeds
    import umap
    t = time.time()
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)          # cosine geometry
    Y = umap.UMAP(n_neighbors=20, min_dist=0.01, random_state=0).fit_transform(PCA(50, random_state=0).fit_transform(Xn))
print(f"UMAP laid out {len(Y):,} vectors in {time.time() - t:.0f} s")
GRID = 16
lo, hi = np.percentile(Y, 0.5, 0), np.percentile(Y, 99.5, 0)
Gx = np.clip(((Y - lo) / (hi - lo) * GRID).astype(int), 0, GRID - 1)
cell = Gx[:, 0] * GRID + Gx[:, 1]
atlas = []
for c in np.unique(cell):
    sel = np.where(cell == c)[0]
    if len(sel) < 8:
        continue
    score = {}
    for a, cl in zip(Att[sel].ravel(), Cls[sel].ravel()):
        score[int(cl)] = score.get(int(cl), 0.0) + float(a)
    best = sorted(score.items(), key=lambda kv: -kv[1])[:3]
    atlas.append({"i": int(c // GRID), "j": int(c % GRID), "n": len(sel), "top": [NAMES[k] for k, _ in best],
                  "v": X[sel].mean(0)})
print(f"{len(atlas)} cells hold 8 or more vectors")

# What the map is made of does not depend on its size: compare the share of cells whose top label is a dog breed,
# and the share that are vehicles, with the lecture's 733 cells.
DOGS = set(NAMES[151:269])
VEHICLES = {"ambulance", "beach wagon", "cab", "convertible", "jeep", "limousine", "minivan", "Model T", "pickup",
            "police van", "racer", "sports car", "minibus", "school bus", "trolleybus", "moving van", "fire engine",
            "garbage truck", "tow truck", "trailer truck", "recreational vehicle", "go-kart", "golfcart", "car wheel"}
dog_ours = np.mean([c["top"][0] in DOGS for c in atlas]); dog_lec = np.mean([n in DOGS for n in LECTURE["atlas"]["top1"]])
veh_ours = np.mean([c["top"][0] in VEHICLES for c in atlas]); veh_lec = np.mean([n in VEHICLES for n in LECTURE["atlas"]["top1"]])
print(f"cells led by a dog breed: ours {dog_ours:.1%}, the lecture's {dog_lec:.1%}; by a vehicle: ours {veh_ours:.1%}, the lecture's {veh_lec:.1%}")
check("share of cells led by a dog breed (any size of atlas; margin 6 points)", dog_ours, dog_lec, tol=0.06)
check("share of cells led by a vehicle (margin 3 points)", veh_ours, veh_lec, tol=0.03)

# neighbourhoods: dog cells sit next to dog cells far more often than chance (the lecture's "regions")
where = {(c["i"], c["j"]): c["top"][0] in DOGS for c in atlas}
pairs = [(where[(i, j)], where[(i + di, j + dj)]) for (i, j) in where for di, dj in ((0, 1), (1, 0)) if (i + di, j + dj) in where]
same = np.mean([a and b for a, b in pairs]) / max(np.mean([a for a, _ in pairs]) ** 2, 1e-9)
print(f"dog cells next to dog cells: {same:.1f} times chance")
check("dogs form a region (neighbours far more often than chance: over 2x)", same > 2, True)

In [ ]:
show = sorted(atlas, key=lambda c: -c["n"])[:48]
icons = draw_directions("mixed4c", np.stack([c["v"] for c in show]), size=96)
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(Y[:, 0], Y[:, 1], s=0.3, color=INK, alpha=0.25, lw=0)
w = (hi - lo) / GRID
for c, im in zip(show, icons):
    x0, y0 = lo[0] + c["i"] * w[0], lo[1] + c["j"] * w[1]
    ax.imshow(im.permute(1, 2, 0).numpy(), extent=(x0, x0 + w[0], y0, y0 + w[1]))
ax.set_xlim(lo[0], hi[0]); ax.set_ylim(lo[1], hi[1]); ax.set_aspect("auto"); ax.axis("off")
ax.set_title("our small atlas of mixed4c: the 48 fullest cells, each its average drawn", fontsize=10)
plt.show()
for c in sorted(show, key=lambda c: (c["i"], c["j"]))[:12]:
    print(f"cell ({c['i']:2d}, {c['j']:2d}), {c['n']:3d} vectors: {', '.join(c['top'])}")

## 6. Whale or shark, and where baseballs land (Plates IX–X)
A **class atlas**: the same idea for just two classes the network can confuse, grey whale and great white shark, from
their ImageNet *training* photos. In each photo keep the 40 of 196 spots that push hardest toward either class; lay
them out with **t-SNE** (another way to put similar vectors close); give each cell the mean push toward each class.

The lecture laid out 20,000 vectors on a 14 × 14 grid. We lay out 6,000 on a 10 × 10 grid. Then drop in vectors from
baseball photos (the lecture's 400 training baseballs; here the 50 baseball test photos, 4 strongest spots each) and
see which cells they resemble most.

In [ ]:
TRAIN = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "train-{:05d}-of-00040.parquet")


def train_photos(c, i):
    path = f"train-{i}.parquet"
    if not os.path.exists(path):
        t = time.time()
        with requests.get(TRAIN.format(i), stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded training file {i} ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t:.0f} s")
    pf = pq.ParquetFile(path)
    lab = pf.read(columns=["label"]).column("label").to_numpy()
    want = np.where(lab == c)[0]
    return [im["bytes"] for im in pf.read(columns=["image"]).column("image").take(want).to_pylist()]


whales_train, sharks = train_photos(WHALE, 5), train_photos(SHARK, 0)
print(len(whales_train), "grey whales and", len(sharks), "great white sharks from the training set")
vecs, atts, src = [], [], []
for c, photos in ((WHALE, whales_train), (SHARK, sharks)):
    for i in range(0, len(photos), 100):
        x = torch.stack([decode(b) for b in photos[i:i + 100]]).to(DEV)
        h, lg = forward_from("mixed4c", x)
        gw, = torch.autograd.grad(lg[:, WHALE].sum(), h, retain_graph=True)
        gs, = torch.autograd.grad(lg[:, SHARK].sum(), h)
        a = torch.stack([(h * gw).sum(1), (h * gs).sum(1)], -1).detach().reshape(-1, 2)
        hv = h.detach().permute(0, 2, 3, 1).reshape(-1, 512)
        top = a.max(1).values.reshape(len(x), -1).topk(40, 1).indices
        keep = torch.zeros(len(x), 196, dtype=torch.bool, device=DEV)
        keep[torch.arange(len(x), device=DEV)[:, None], top] = True
        keep = keep.reshape(-1)
        vecs.append(hv[keep].cpu()); atts.append(a[keep].cpu()); src += [c] * int(keep.sum())
Xv, Av, src = torch.cat(vecs).numpy(), torch.cat(atts).numpy(), np.array(src)
check("vectors kept: 40 spots x the whale and shark photos", len(Xv), 40 * (len(whales_train) + len(sharks)))
sub = np.random.default_rng(0).permutation(len(Xv))[:6000]
Xv, Av, src = Xv[sub], Av[sub], src[sub]

from sklearn.manifold import TSNE
t = time.time()
Yw = TSNE(2, metric="cosine", init="pca", random_state=0, perplexity=40).fit_transform(Xv)
print(f"t-SNE laid out {len(Yw):,} vectors in {time.time() - t:.0f} s")
g = 10
lo_w, hi_w = np.percentile(Yw, 1, 0), np.percentile(Yw, 99, 0)
Gw = np.clip(((Yw - lo_w) / (hi_w - lo_w) * g).astype(int), 0, g - 1)
cw = Gw[:, 0] * g + Gw[:, 1]
wcells = []
for c in np.unique(cw):
    sel = np.where(cw == c)[0]
    if len(sel) < 10:
        continue
    wcells.append({"i": int(c // g), "j": int(c % g), "n": len(sel), "attr_whale": float(Av[sel, 0].mean()),
                   "attr_shark": float(Av[sel, 1].mean()), "shark_share": float((src[sel] == SHARK).mean()), "v": Xv[sel].mean(0)})
print(f"{len(wcells)} cells; {sum(c['attr_shark'] > c['attr_whale'] for c in wcells)} push harder toward shark")

In [ ]:
labels = np.concatenate([pq.ParquetFile(f"val-{s}.parquet").read(columns=["label"]).column("label").to_numpy() for s in (0, 1)])
bb_ids = [int(i) for i in np.where(labels == BASEBALL)[0]]
got = rows(bb_ids)
xb = torch.stack([decode(got[i][0]) for i in bb_ids]).to(DEV)
with torch.no_grad(), Taps(["mixed4c"]) as T:
    net(prep(xb))
hb = T["mixed4c"].permute(0, 2, 3, 1).reshape(len(xb), -1, 512)
topb = hb.norm(dim=-1).topk(4, 1).indices
Bv = torch.gather(hb, 1, topb[..., None].expand(-1, -1, 512)).reshape(-1, 512).cpu().numpy()
V = np.stack([c["v"] for c in wcells])
cos = (Bv / np.linalg.norm(Bv, axis=1, keepdims=True)) @ (V / np.linalg.norm(V, axis=1, keepdims=True)).T
landed = np.bincount(cos.argmax(1), minlength=len(V))
order = np.argsort(-landed)
share2 = landed[order[:2]].sum() / landed.sum()
for k in order[:3]:
    c = wcells[k]
    print(f"cell ({c['i']}, {c['j']}): {landed[k]} of {len(Bv)} baseball vectors; {c['shark_share']:.1%} of its own from shark photos; "
          f"pushes toward shark {c['attr_shark']:.2f}, whale {c['attr_whale']:.2f}")
want = LECTURE["whale"]["top_cell"]
print(f"the lecture's top cell: {want['baseball_hits']} of 1,600; {want['frac_from_shark_photos']:.1%} from sharks; "
      f"shark {want['attr_shark']:.2f}, whale {want['attr_whale']:.2f}; top two cells hold {LECTURE['whale']['share_top2']:.0%}")
best = wcells[order[0]]
check("the cell most baseballs land in pushes toward shark, not whale", best["attr_shark"] > best["attr_whale"], True)
check("and it is made mostly of shark photos (over 80%)", best["shark_share"] > 0.8, True)
check("baseballs crowd into few cells: the top two hold over 25% (lecture, on its finer map: 45%)", share2 > 0.25, True)
stitch = draw_directions("mixed4c", V[order[:2]], size=128)
fig, axs = plt.subplots(1, 3, figsize=(7.5, 2.7))
axs[0].imshow(xb[0].permute(1, 2, 0).cpu()); axs[0].set_title("a baseball test photo", fontsize=9)
for a, im, k in zip(axs[1:], stitch, order[:2]):
    a.imshow(im.permute(1, 2, 0)); a.set_title(f"cell ({wcells[k]['i']}, {wcells[k]['j']}): {landed[k]} baseball vectors", fontsize=9)
for a in axs:
    a.axis("off")
plt.show()

## 7. The attack (Plate XI), at the lecture's full scale
If those cells vote shark, a baseball should push a whale toward shark. Take every grey whale photo the network gets
right (the 1,150 training whales and the 50 test whales), cut a round disc from the centre of each of the 5 test
photos the network is surest are baseballs, and paste it in each corner (20 placements), at several sizes. An image
made to fool a network is an **adversarial patch**; Lecture 14 returns to them.

In [ ]:
val_whales = [int(i) for i in np.where(labels == WHALE)[0]]
got = rows(val_whales)
Xw = torch.stack([decode(b) for b in whales_train] + [decode(got[i][0]) for i in val_whales])
check("grey whale photos", len(Xw), LECTURE["patch"]["n_whales"])


def predict(X, bs=200):
    out = []
    for i in range(0, len(X), bs):
        with torch.no_grad(), Taps(["softmax2_pre_activation_matmul"]) as T:
            net(prep(X[i:i + bs].to(DEV)))
        out.append(T["softmax2_pre_activation_matmul"][:, PERM].softmax(1).cpu())
    return torch.cat(out)

pw = predict(Xw)
ok = pw.argmax(1) == WHALE
Xw = Xw[ok]
check("whales the network gets right", int(ok.sum()), LECTURE["patch"]["n_ok"])
check("their mean chance of shark, before", float(pw[ok, SHARK].mean()), LECTURE["patch"]["clean_p_shark"], tol=0.0001)


def best_photos(c, n=5):
    ids_c = [int(i) for i in np.where(labels == c)[0]]
    got_c = rows(ids_c)
    X = torch.stack([decode(got_c[i][0]) for i in ids_c])
    o = predict(X)[:, c].argsort(descending=True)[:n]
    return X[o], [ids_c[int(i)] for i in o]

SRC = {}
for name in ("baseball", "tennis ball", "golf ball"):
    SRC[name], picked = best_photos(NAMES.index(name))
    want = [int(p.split("val_")[1][:5]) for p in LECTURE["patch"]["sources"][name]]
    check(f"the 5 test photos surest to be a {name}", picked, want)

yy, xx = torch.meshgrid(torch.linspace(-1, 1, 224), torch.linspace(-1, 1, 224), indexing="ij")
disc = ((yy ** 2 + xx ** 2) <= 0.62 ** 2).float()


def paste(X, src, s, corner):
    P = F.interpolate((src * disc)[None], size=(s, s), mode="bilinear", align_corners=False)[0]
    Mk = F.interpolate(disc[None, None], size=(s, s), mode="bilinear", align_corners=False)[0]
    Y = X.clone()
    y0 = 8 if corner in (0, 1) else 224 - s - 8
    x0 = 8 if corner in (0, 2) else 224 - s - 8
    Y[:, :, y0:y0 + s, x0:x0 + s] = Y[:, :, y0:y0 + s, x0:x0 + s] * (1 - Mk) + P * Mk
    return Y


def attack(name, s, cls_patch):
    flips, to_patch, dlog, any_flip, example = [], [], [], torch.zeros(len(Xw), dtype=torch.bool), None
    for k in range(5):
        for corner in range(4):
            p = predict(paste(Xw, SRC[name][k], s, corner))
            f = p.argmax(1) == SHARK
            any_flip |= f
            flips.append(f.float().mean().item()); to_patch.append((p.argmax(1) == cls_patch).float().mean().item())
            dlog.append((p[:, SHARK].clamp_min(1e-9).log() - pw[ok, SHARK].clamp_min(1e-9).log()).mean().item())
            if example is None and f.any():
                example = (k, corner, int(f.nonzero()[0]))
    return {"to_shark": np.mean(flips), "to_shark_best_case_any": any_flip.float().mean().item(),
            "to_patch_class": np.mean(to_patch), "mean_dlog_p_shark": np.mean(dlog)}, example


t = time.time()
R32, ex = attack("baseball", 32, BASEBALL)
k, corner, i = ex
before, after = predict(Xw[i:i + 1])[0], predict(paste(Xw[i:i + 1], SRC["baseball"][k], 32, corner))[0]
print("the first whale a 32-px baseball flips:")
print("  before:", [(NAMES[int(c)], round(float(v), 3)) for v, c in zip(*before.topk(3))])
print("  after: ", [(NAMES[int(c)], round(float(v), 3)) for v, c in zip(*after.topk(3))])
want = LECTURE["patch"]["example"]
check("the example: which baseball photo, which corner", [k, corner], [want["source"], want["corner"]])
check("before: grey whale", float(before[WHALE]), dict(want["p_before"])["grey whale"], tol=0.001)
check("after: great white shark", float(after[SHARK]), dict(want["p_after"])["great white shark"], tol=0.001)
check("after: the answer", NAMES[int(after.argmax())], want["p_after"][0][0])
fig, axs = plt.subplots(1, 2, figsize=(5.6, 2.9))
axs[0].imshow(Xw[i].permute(1, 2, 0)); axs[0].set_title(f"grey whale {before[WHALE]:.1%}", fontsize=9)
axs[1].imshow(paste(Xw[i:i + 1], SRC["baseball"][k], 32, corner)[0].permute(1, 2, 0)); axs[1].set_title(f"great white shark {after[SHARK]:.1%}", fontsize=9)
for a in axs:
    a.axis("off")
plt.show()

In [ ]:
runs = {"baseball|32": R32, "baseball|96": attack("baseball", 96, BASEBALL)[0], "baseball|128": attack("baseball", 128, BASEBALL)[0],
        "tennis ball|128": attack("tennis ball", 128, NAMES.index("tennis ball"))[0]}
print(f"{len(runs) * 20} placements over {len(Xw):,} whales in {time.time() - t:.0f} s")
print(f"{'patch':16s} {'one placement':>14s} {'best of 20':>11s} {'becomes the patch':>18s} {'log-odds of shark':>18s}")
for kname, r in runs.items():
    print(f"{kname:16s} {r['to_shark']:14.2%} {r['to_shark_best_case_any']:11.2%} {r['to_patch_class']:18.1%} {r['mean_dlog_p_shark']:+18.2f}")
W = LECTURE["patch"]["runs"]
for kname, r in runs.items():
    for m, tol in (("to_shark", 0.0005), ("to_shark_best_case_any", 0.002), ("to_patch_class", 0.002), ("mean_dlog_p_shark", 0.01)):
        check(f"{kname}: {m.replace('_', ' ')}", r[m], W[kname][m], tol=tol)
print(f"at 96 px the odds of shark rise {math.exp(runs['baseball|96']['mean_dlog_p_shark']):.0f}x")

About 2 in 5 whales can be turned into a shark by a baseball in the best of 20 spots, as the paper found; a single
placement does it only 1 time in 20, and big patches mostly turn the answer into "baseball". A tennis ball almost never
makes a shark: the shortcut is the baseball's red seam, which the map files next to shark teeth.

## Your turn
1. **The second car detector.** Which other mixed4c neuron does dissection call a car, with the next best IoU?
   *Hint: sort `DISSECT["mixed4c"]`.*
2. **The last layer's odd favourite.** In mixed5b, which label do the most detectors share? *Hint: count the labels of
   channels with IoU over 0.04.*
3. **A golf ball.** Paste golf balls at 128 px. How often does a whale become a shark, best of 20? *Hint:
   `attack("golf ball", 128, ...)`.*
4. **Not just whales.** Paste the first baseball at 64 px in the bottom-right corner of 2,000 random test photos
   (seed 0, no whales or sharks). How often is the answer shark? *Hint: `paste(X, SRC["baseball"][0], 64, 3)`.*

In [ ]:
# Your turn 1
# TODO

In [ ]:
# Your turn 2
# TODO

In [ ]:
# Your turn 3
# TODO

In [ ]:
# Your turn 4
# TODO

## Solutions

In [ ]:
# 1. The second car detector
lab, val = DISSECT["mixed4c"]
cars = [int(c) for c in np.argsort(-val) if ADE_NAMES[lab[c]].split(",")[0] == "car"]
print(f"4c:{cars[1]}: car, IoU {val[cars[1]]:.3f}")
check("the second car detector of mixed4c", [cars[1], round(float(val[cars[1]]), 3)], LECTURE["dissect"]["mixed4c"]["top"][1][::2])

In [ ]:
# 2. The last layer's odd favourite
from collections import Counter
lab, val = DISSECT["mixed5b"]
common = Counter(ADE_NAMES[c].split(",")[0] for c in lab[val > 0.04]).most_common(1)[0]
print(f"mixed5b's most common label: {common[0]}, {common[1]} detectors (probably green, felt-like regions)")
check("mixed5b's most common label", list(common), list(LECTURE["dissect"]["mixed5b"]["counts_top"]))

In [ ]:
# 3. A golf ball
r, _ = attack("golf ball", 128, NAMES.index("golf ball"))
print(f"golf ball, 128 px: shark in the best of 20 for {r['to_shark_best_case_any']:.1%} of whales (also white and stitched-looking)")
check("golf ball 128: best of 20", r["to_shark_best_case_any"], LECTURE["patch"]["runs"]["golf ball|128"]["to_shark_best_case_any"], tol=0.002)

In [ ]:
# 4. Not just whales
others = [int(i) for i in np.random.default_rng(0).choice(50000, 2000, replace=False) if labels[i] not in (WHALE, SHARK)]
got = rows(others)
Xo = torch.stack([decode(got[i][0]) for i in others])
p = predict(paste(Xo, SRC["baseball"][0], 64, 3))
rate = float((p.argmax(1) == SHARK).float().mean())
print(f"the same baseball on {len(Xo):,} other photos: shark {rate:.2%} of the time")
check("baseball on other photos, 64 px: shark", rate, LECTURE["patch"]["runs"]["baseball_on_other_photos|64"]["to_shark"], tol=0.0005)
print(f"notebook ran in {time.time() - T0:.0f} s")